<a href="https://colab.research.google.com/github/ryuboku555/janken-roulatte/blob/main/JA%E3%81%8A%E7%9F%A5%E3%82%89%E3%81%9B.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [ ]:
import requests
from bs4 import BeautifulSoup
import urllib.parse
import re
import pandas as pd
import gspread
from google.auth import default
from google.colab import auth

# --- ステップ1: 認証 ---
auth.authenticate_user()
creds, _ = default()
gc = gspread.authorize(creds)

# --- ステップ2: 関数（日付処理を強化） ---
def extract_news(html_content, base_url, site_name):
    try:
        html = html_content.decode('utf-8')
    except UnicodeDecodeError:
        html = html_content.decode('shift_jis', errors='ignore')
    soup = BeautifulSoup(html, 'html.parser')
    news_items = []
    seen_links = set()
    date_pattern = re.compile(r'(202\d|令和\d+|R\d+)[./年-]\s*(\d{1,2})[./月-]\s*(\d{1,2})[日]*')

    for a in soup.find_all('a', href=True):
        href = a['href']
        if href.startswith('#') or href.endswith(('.jpg', '.png', '.gif')): continue
        link = urllib.parse.urljoin(base_url, href)
        if link in seen_links: continue
        title = a.get_text(strip=True)
        if not title or len(title) <= 2: continue

        date_str = None
        match = date_pattern.search(a.get_text(separator=' '))
        if not match and a.parent:
            match = date_pattern.search(a.parent.get_text(separator=' '))

        if match:
            y, m, d = match.group(1), match.group(2), match.group(3)
            # 令和/R を西暦に変換
            if "令和" in y or "R" in y:
                y_num = int(re.sub(r'\D', '', y))
                y = str(2018 + y_num)
            norm_date = f"{y}-{int(m):02d}-{int(d):02d}"
            news_items.append({'日付': norm_date, 'JA名': site_name, 'タイトル': title, 'リンクURL': link})
            seen_links.add(link)
    return news_items

# --- ステップ3: 取得実行 ---
sites = [
    ('JAセレサ川崎', 'https://www.jaceresa.or.jp/'),
    ('JA横浜', 'https://ja-yokohama.or.jp/'),
    ('JAさがみ', 'https://ja-sagami.or.jp/'),
    ('JAさがみはら', 'https://www.ja-sagamiharashi.or.jp/')
]
all_news = []
for name, url in sites:
    try:
        res = requests.get(url, headers={'User-Agent': 'Mozilla/5.0'}, timeout=15)
        all_news.extend(extract_news(res.content, url, name))
    except: continue

# --- ステップ4: スプレッドシートへ「追記」 ---
if all_news:
    sh = gc.open("JAお知らせ一覧")
    worksheet = sh.worksheet("シート1")

    # すでにシートにあるURLを取得して、重複を防ぐ
    existing_urls = worksheet.col_values(4) # D列がURL

    new_data_to_add = []
    for item in all_news:
        if item['リンクURL'] not in existing_urls:
            new_data_to_add.append([item['日付'], item['JA名'], item['タイトル'], item['リンクURL']])

    if new_data_to_add:
        # 日付で昇順（古い順）に並べ替えてから追加（新しいのが下に来る）
        new_data_to_add.sort(key=lambda x: x[0])
        worksheet.append_rows(new_data_to_add)
        print(f"{len(new_data_to_add)}件の新しい記事を追加しました！")
    else:
        print("新しい記事はありませんでした。")

6件の新しい記事を追加しました！
